# Installation and Imports

In [ ]:
!pip install -q -U transformers
!pip install -q -U datasets
!pip install -q -U bitsandbytes
!pip install -q -U trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 118.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 18.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12 

In [ ]:
from transformers import (pipeline, AutoModelForCausalLM, AutoTokenizer,
                          BitsAndBytesConfig, TrainingArguments, Trainer)
from datasets import load_dataset, Dataset
import bitsandbytes as bnb
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
import torch

import numpy as np
import random
import re
from pprint import pprint

from google.colab import drive, userdata


In [ ]:
drive.mount('/content/drive')
project_path = "/content/drive/MyDrive/DATASCI 266/Final Project"

Mounted at /content/drive


# DataSets

## Basic Arithmetic

In [ ]:
# def gen_arithmetic_qa():
#     a, b, c, d, e, f = np.random.randint(0, 30, size=6)
#     ops = random.choices(['+', '-', '*'], k=5)

#     expr = f"{a}{ops[0]}{b}{ops[1]}{c}{ops[2]}{d}{ops[3]}{e}{ops[4]}{f}"
#     question = f"{expr}=?"
#     answer = eval(expr)

#     return question, answer

# # Create Arithmetic dataset
# arithmetics_dict = {'question': [], 'answer': []}
# random.seed(42)

# for _ in range(1000):
#     q, a = gen_arithmetic_qa()
#     arithmetics_dict['question'].append(q)
#     arithmetics_dict['answer'].append(a)

# Arithmetics = Dataset.from_dict(arithmetics_dict)
# Arithmetics.to_json(f"{project_path}/data/arithmetics.json")

In [ ]:
Arithmetics = load_dataset("json", data_files=f"{project_path}/data/arithmetics.json")
print("Question:", Arithmetics['train']['question'][0])
print("Answer:", Arithmetics['train']['answer'][0])

Generating train split: 0 examples [00:00, ? examples/s]

Question: 9-0+14+18+15*23=?
Answer: 386


## Grade School Math

In [ ]:
GSM = load_dataset("openai/gsm8k", "main")
GSM = GSM['test'].train_test_split(test_size=0.2)
GSM

README.md:   0%|          | 0.00/7.94k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 1055
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 264
    })
})

## High Shool Math

In [ ]:
HSM = load_dataset("HuggingFaceH4/MATH-500")
HSM = HSM['test'].train_test_split(test_size=0.2)
HSM

README.md:   0%|          | 0.00/412 [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/447k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['problem', 'solution', 'answer', 'subject', 'level', 'unique_id'],
        num_rows: 400
    })
    test: Dataset({
        features: ['problem', 'solution', 'answer', 'subject', 'level', 'unique_id'],
        num_rows: 100
    })
})

# LoRA

In [ ]:
"""
Initialize the pipeline with bitsandbytes quantization
"""
# Configure bitsandbytes for 4-bit quantization
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
def train_lora(model_id, dataset):
    # Load Base Model
    base_model = AutoModelForCausalLM.from_pretrained(model_id,
                                                      torch_dtype=torch.bfloat16,
                                                      quantization_config=quantization_config,
                                                      low_cpu_mem_usage=True,
                                                    #   use_cache=False,
                                                      attn_implementation='eager')
    base_model = prepare_model_for_kbit_training(base_model)
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    # Prepare LoRA Model
    lora_config = LoraConfig(
        task_type="CAUSAL_LM",
        r=8,
        lora_dropout=0.1,
        target_modules=["q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"]
    )

    lora_model = prepare_model_for_kbit_training(base_model)
    lora_model = get_peft_model(lora_model, lora_config)

    lora_model.print_trainable_parameters()

    # Load dataset and define formatting function
    if dataset == "gsm":
        def format_func(example):
            text = f"""You are a helpful assistant. Solve the question below and provide the answer at the end.
            ### Question: {example['question']}
            ### Answer: \\boxed{{{example['answer']}}}"""
            return text

        train_dataset = GSM['train']
        eval_dataset = GSM['test']
    elif dataset == "hsm":
        def format_func(example):
            text = f"""You are a helpful assistant. Solve the question below and provide the answer at the end.
            ### Problem: {example['problem']}
            ### Solution: {example['solution']}
            ### Answer: \\boxed{{{example['answer']}}}"""
            return text

        train_dataset = HSM['train']
        eval_dataset = HSM['test']
    else:
        raise ValueError("Invalid dataset name")

    # Training
    training_args = TrainingArguments(
        output_dir=f"{project_path}/lora/{model_id}-qlora-{dataset}",
        learning_rate=1e-3,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=3,
        weight_decay=0.01,
        bf16=True,
        save_total_limit=4,
        gradient_checkpointing=True,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        report_to="none",
        push_to_hub=True
    )

    trainer = SFTTrainer(
        model=lora_model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        peft_config=lora_config,
        formatting_func=format_func
        )

    trainer.train()

    # Delete model and release gpu ram
    del lora_model
    torch.cuda.empty_cache()

    return f"bosen-chia-1/{model_id}-qlora-{dataset}"

### Llama + GSM

In [ ]:
# model_id = "meta-llama/Llama-3.2-3B-Instruct"
# dataset = "gsm"

# lora_model_id = train_lora(model_id, dataset)

### Llama + HSM

In [ ]:
# model_id = "meta-llama/Llama-3.2-3B-Instruct"
# dataset = "hsm"

# lora_model_id = train_lora(model_id, dataset)

### Gemma + GSM

In [ ]:
# model_id = "google/gemma-3-4b-it"
# dataset = "gsm"

# lora_model_id = train_lora(model_id, dataset)

### Gemma + HSM

In [ ]:
# model_id = "google/gemma-3-4b-it"
# dataset = "hsm"

# lora_model_id = train_lora(model_id, dataset)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.64G [00:00<?, ?B/s]

...

Applying chat template to eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Epoch,Training Loss,Validation Loss
1,No log,0.694641
2,No log,0.721037
3,No log,0.833709


In [ ]:
# LoRA Inference
model_id = "bosen-chia-1/Llama-3.2-3B-Instruct-qlora-GSM"

lora_pipe = pipeline(
    "text-generation",
   model=model_id,
   model_kwargs={"torch_dtype": torch.bfloat16,
                 "quantization_config": quantization_config,
                 "low_cpu_mem_usage": True,
                 "use_cache": False},
   device_map="auto",
   trust_remote_code=True
)


adapter_config.json:   0%|          | 0.00/803 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

...

adapter_model.safetensors:   0%|          | 0.00/48.7M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Device set to use cuda:0


In [ ]:
prompt = [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": GSM['test']['question'][0]}
        ]

result = lora_pipe(prompt, max_new_tokens=300)
print("LoRA Llama response:")
pprint(result[0]['generated_text'][-1]['content'])

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


LoRA Llama response:
('Let x be the measure of the smallest angle.\n'
 'The largest angle is 2x.\n'
 'There are also two right angles, which is 2*90 = 180 degrees.\n'
 'The sum of all angles in a quadrilateral is 360 degrees.\n'
 'So, 2x + x + 180 + 180 = 360\n'
 '3x + 360 = 360\n'
 '3x = 0\n'
 'x = 0\n'
 'The largest angle is 2*0 = 0 degrees.\n'
 '#### 0}')


In [ ]:
HSM['test']['problem'][0]

'In a convex quadrilateral, the measure of the largest angle is twice the measure of the smallest angle, and the other two angles are both right angles. How many degrees are in the largest angle?'